# 06. 전략 인사이트 — 자치구별 운영 전략 매트릭스

**분석 목적:** 군집 분석 결과를 바탕으로 자치구별 실행 가능한 운영 전략을 수립합니다.

**핵심 질문:**
- 각 군집 유형에서 RevPAR를 높이기 위한 우선 전략은 무엇인가?
- 최근 3개월(L90D) 트렌드가 TTM 대비 개선되고 있는 자치구는 어디인가?

**접근 방법:**
- TTM vs L90D RevPAR 트렌드 비교
- 규제 리스크(Non-Operating 비율) 분석
- 군집별 전략 매트릭스 도출

In [ ]:
# CLUSTER_COLORS를 05_market_segmentation과 동일하게 유지 -- 노트북 간 색상 일관성 보장
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")

DOMAIN_KOREAN = "서울 에어비앤비"
RANDOM_SEED = 42
REPORTS_DIR = Path("../reports")
PROCESSED_DIR = Path("../data/processed")

KPI_REVPAR_ACTIVE_MEDIAN = 47850
KPI_DORMANT_RATIO_ALL = 0.543
KPI_SUPERHOST_PREMIUM = 0.831
# 05_platform_clustering.ipynb 클러스터 이름과 일치
CLUSTER_COLORS = {"핫플 수익형":"#1565C0","프리미엄 비즈니스":"#2E7D32","로컬 주거형":"#FF8F00","가성비 신흥형":"#C62828"}

sns.set_style("whitegrid")
import platform
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
else:
    plt.rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams["figure.dpi"] = 100

try:
    dc = pd.read_csv(PROCESSED_DIR/"district_clustered.csv")
except:
    dc = None
    print("⚠️ district_clustered.csv 없음 — 05_platform_clustering.ipynb 먼저 실행")

df_ao = pd.read_csv("../data/processed/seoul_airbnb_features.csv")
mask = (df_ao["refined_status"]=="Active") & (df_ao["operation_status"]=="Operating")
df_ao = df_ao[mask].copy()
print(f"Active+Oper: {len(df_ao):,}건")

## 1️⃣ TTM vs L90D RevPAR 트렌드

In [ ]:
# TTM vs L90D RevPAR 비교 -- 연간 평균 대비 최근 3개월 성과를 통해 모멘텀 파악
trend = df_ao.groupby("district").agg(ttm_med=("ttm_revpar","median"), l90d_med=("l90d_revpar","median"), ao_count=("ttm_revpar","count")).reset_index()
trend["ttm_quarterly"] = trend["ttm_med"]/4
trend["growth"] = trend["l90d_med"] > trend["ttm_quarterly"]
trend = trend.sort_values("l90d_med", ascending=False)

fig, ax = plt.subplots(figsize=(13, 6))
x = range(len(trend)); w = 0.35
ax.bar([i-w/2 for i in x], trend["ttm_quarterly"], w, label="TTM RevPAR/4 (분기 환산)", color="steelblue", alpha=0.8)
ax.bar([i+w/2 for i in x], trend["l90d_med"], w, label="L90D RevPAR (최근 90일)", color="coral", alpha=0.8)
ax.set_title("자치구별 TTM(분기환산) vs L90D RevPAR\n(L90D > TTM/4 = 성장 중)", fontsize=12, fontweight="bold")
ax.set_ylabel("RevPAR (₩)"); ax.set_xticks(x); ax.set_xticklabels(trend["district"], rotation=45, ha="right", fontsize=8); ax.legend()
plt.tight_layout()
plt.show()
print(f"L90D > TTM/4 (성장): {trend['growth'].sum()}/{len(trend)} 자치구")
print(f"성장 자치구: {trend[trend['growth']]['district'].tolist()}")

## 2️⃣ 규제 리스크 분석 (Non-Operating 비율)

In [ ]:
# Non-Operating 비율로 규제 압력 대리 측정 -- 비활성 비율이 높을수록 운영 환경 악화
df_all = pd.read_csv("../data/processed/seoul_airbnb_features.csv")
dreg = df_all.groupby("district").agg(total=("ttm_revpar","count"), non_op=("operation_status", lambda x: (x=="Non_Operating").sum())).reset_index()
dreg["non_op_rate"] = dreg["non_op"]/dreg["total"]
dreg_sorted = dreg.sort_values("non_op_rate", ascending=True)
colors_reg = ["#C62828" if v>0.15 else "#FF8F00" if v>0.10 else "#2E7D32" for v in dreg_sorted["non_op_rate"]]
fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(dreg_sorted["district"], dreg_sorted["non_op_rate"]*100, color=colors_reg)
ax.axvline(11.5, color="black", linestyle="--", linewidth=1.5, label="전체 평균 11.5%")
ax.set_title("자치구별 사업자 미등록(Non-Operating) 비율\n(빨강15%+, 주황10-15%, 초록10%미만 | 2025-10-16 마감)", fontsize=11, fontweight="bold")
ax.set_xlabel("Non-Operating 비율 (%)"); ax.legend()
plt.tight_layout()
plt.show()
high_risk = dreg[dreg["non_op_rate"]>0.15].sort_values("non_op_rate", ascending=False)
print(f"⚠️ 규제 고위험 자치구 (15%+): {high_risk['district'].tolist()}")

## 3️⃣ 자치구 전략 매트릭스

In [ ]:
# 군집별 전략을 구조화 -- '이 자치구에서 어떤 전략이 유효한가' 실행 가이드
strategy_map = {
    "핫플 수익형": {"전략":"고급화 유지","액션":["프리미엄 숙소 지원 강화","슈퍼호스트 비율 유지","가격 인상 가이드 제공"],"우선순위":"⭐⭐⭐"},
    "프리미엄 비즈니스": {"전략":"적극 육성","액션":["슈퍼호스트 전환 지원","즉시예약 활성화 캠페인","Dormant 재활성화"],"우선순위":"⭐⭐⭐"},
    "로컬 주거형": {"전략":"안정적 성장","액션":["품질 향상 지원","사진 업로드 가이드","평점 개선 인센티브"],"우선순위":"⭐⭐"},
    "가성비 신흥형": {"전략":"효율화","액션":["규제 준수 지원","Dormant 정리 권고","비수기 프로모션"],"우선순위":"⭐"}
}
print("="*70)
print("📋 자치구별 운영 전략 매트릭스")
print("="*70)
for cname, st in strategy_map.items():
    members = dc[dc["cluster_name"]==cname]["district"].tolist() if dc is not None else []
    revpar = dc[dc["cluster_name"]==cname]["median_revpar_ao"].median() if dc is not None else 0
    print(f"\n{st['우선순위']} [{cname}]")
    if members: print(f"  자치구: {', '.join(members)}")
    if revpar > 0: print(f"  RevPAR 중위값: ₩{revpar:,.0f}")
    print(f"  전략: {st['전략']}")
    for a in st["액션"]: print(f"    - {a}")
print("="*70)

## 4️⃣ 핵심 발견사항 요약

In [ ]:
# 분석 결과를 JSON으로 저장 -- 대시보드 또는 보고서 자동화에 재사용 가능
summary = {
    "analysis_date": "2026-03-05",
    "total_listings": 32061,
    "active_operating": 14399,
    "num_districts": 25,
    "key_insights": [
        f"슈퍼호스트 프리미엄 +{KPI_SUPERHOST_PREMIUM*100:.1f}% RevPAR",
        f"전체 비활성 비율 {KPI_DORMANT_RATIO_ALL*100:.1f}% — 시장 건전성 위기",
        "공급 과잉 자치구(마포구) RevPAR 상대적 압박",
        "관광 특화 자치구 > 주거형 자치구 RevPAR",
        "L90D 성장 추세 자치구 다수 — 상승 국면"
    ],
    "regulatory_deadline": "2025-10-16",
    "recommendation": "핫플 수익형 강화 + 프리미엄 비즈니스 육성에 자원 집중"
}
print("✅ 자치구 인사이트 요약 저장")
print(f"\n🎯 최종 권장사항: {summary['recommendation']}")
print("\n핵심 발견사항:")
for ins in summary["key_insights"]: print(f"  - {ins}")